# ⛓️ 3.3 LCEL Deepdive

## Learning Objectives
In this notebook, you will learn:
1. **The `|` operator** - how Python operator overloading (`__or__`) is what makes `prompt | model | parser` work
2. **Build a Runnable from scratch** - a hand-rolled `CRunnable`/`CRunnableSequence` pair that implements the same `__or__` composition LangChain uses internally
3. **LangChain's built-in Runnables** - `RunnablePassthrough`, `RunnableLambda`, and `RunnableParallel`, invoked individually and composed together
4. **Nested chains** - putting a sequential chain inside one branch of a `RunnableParallel`
5. **`.assign()` and chain composition** - adding computed keys to a running dict, and piping one finished chain into another

## Prerequisites
- Completion of `3.1_LCEL_Introduction.ipynb` and `3.2_Runnables.ipynb`
- An `OPENAI_API_KEY` set in a `.env` file at the project root
- Familiarity with Python dunder methods (`__add__`, `__or__`) is helpful but not required — this notebook builds that intuition from scratch

> **Source**: This notebook's exercises are adapted from the Udemy course [LangChain in Action](https://www.udemy.com/course/langchain-in-action-develop-llm-powered-applications/).

Link: https://www.udemy.com/course/langchain-in-action-develop-llm-powered-applications/

### LCEL Deepdive

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and load .env
# ============================================================================
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

In [ ]:
# ============================================================================
# BASIC LCEL CHAIN: prompt | model | output_parser
# ============================================================================
prompt = ChatPromptTemplate.from_template("tell me a short joke about {topic}")
model = ChatOpenAI(model="gpt-4o-mini")
output_parser = StrOutputParser()

chain = prompt | model | output_parser

chain.invoke({"topic": "ice cream"})

In [ ]:
# ============================================================================
# PROMPT INVOKE: Inspect the PromptValue produced by prompt.invoke()
# ============================================================================
prompt.invoke({"topic": "ice cream"})

In [ ]:
# ============================================================================
# RAW MESSAGE INVOKE: Call the model directly with a HumanMessage
# ============================================================================
from langchain_core.messages.human import HumanMessage

messages = [HumanMessage(content='tell me a short joke about ice cream')]
model.invoke(messages)

In [ ]:
# ============================================================================
# OUTPUT PARSER STANDALONE: Parse an AIMessage directly
# ============================================================================
from langchain_core.messages import AIMessage

ai_msg = AIMessage(content='Why did the ice cream truck break down? It had too many "scoops"!')

output_parser.invoke(ai_msg)

### Operator Overloading

In [ ]:
# ============================================================================
# OPERATOR OVERLOADING: Python's built-in + operator
# ============================================================================
print(2+4)

In [ ]:
# ============================================================================
# OPERATOR OVERLOADING: Same addition via the dunder method
# ============================================================================
result = (2).__add__(4)

print(result)

In [ ]:
# ============================================================================
# OPERATOR OVERLOADING: Custom __add__ on a toy class
# ============================================================================
class StupidAdder:
    def __init__(self, number):
        self.number = number

    def __add__(self, other):
        return StupidAdder(self.number + other.number + 42)

    def __str__(self):
        return str(self.number)

In [ ]:
# ============================================================================
# OPERATOR OVERLOADING: Using the custom + operator
# ============================================================================
first = StupidAdder(5)
second = StupidAdder(10)

print(first + second)

### What is this "|" in Python?

In [ ]:
# ============================================================================
# CUSTOM RUNNABLE: Hand-rolled Runnable/RunnableSequence using __or__
# ============================================================================
from abc import ABC, abstractmethod

class CRunnable(ABC):
    def __init__(self):
        self.next = None

    @abstractmethod
    def process(self, data):
        """
        This method must be implemented by subclasses to define
        data processing behavior.
        """
        pass

    def invoke(self, data):
        processed_data = self.process(data)
        if self.next is not None:
            return self.next.invoke(processed_data)
        return processed_data

    def __or__(self, other):
        return CRunnableSequence(self, other)

class CRunnableSequence(CRunnable):
    def __init__(self, first, second):
        super().__init__()
        self.first = first
        self.second = second

    def process(self, data):
        pass

    def invoke(self, data):
        first_result = self.first.invoke(data)
        return self.second.invoke(first_result)


In [ ]:
# ============================================================================
# CUSTOM RUNNABLE: Example steps built on CRunnable
# ============================================================================
class AddTen(CRunnable):
    def process(self, data):
        print("AddTen: ", data)
        return data + 10

class MultiplyByTwo(CRunnable):
    def process(self, data):
        print("Multiply by 2: ", data)
        return data * 2

class ConvertToString(CRunnable):
    def process(self, data):
        print("Convert to string: ", data)
        return f"Result: {data}"

In [ ]:
# ============================================================================
# CUSTOM RUNNABLE: Compose steps with the | operator
# ============================================================================
a = AddTen()
b = MultiplyByTwo()
c = ConvertToString()

chain = a | b | c

In [ ]:
# ============================================================================
# CUSTOM RUNNABLE: Execute the composed chain
# ============================================================================
result = chain.invoke(10)
print(result)

### Runnables from LangChain

In [ ]:
# ============================================================================
# LANGCHAIN RUNNABLES: Import RunnablePassthrough/RunnableLambda/RunnableParallel
# ============================================================================
from langchain_core.runnables import RunnablePassthrough, RunnableLambda, RunnableParallel

In [ ]:
# ============================================================================
# RUNNABLEPASSTHROUGH: Chain of pass-throughs
# ============================================================================
chain = RunnablePassthrough() | RunnablePassthrough () | RunnablePassthrough ()
chain.invoke("hello")

In [ ]:
# ============================================================================
# RUNNABLELAMBDA: Define a function to wrap
# ============================================================================
def input_to_upper(input: str):
    output = input.upper()
    return output

In [ ]:
# ============================================================================
# RUNNABLELAMBDA: Insert the wrapped function into a chain
# ============================================================================
chain = RunnablePassthrough() | RunnableLambda(input_to_upper) | RunnablePassthrough()
chain.invoke("hello")

In [ ]:
# ============================================================================
# RUNNABLEPARALLEL: Build a parallel branch dictionary
# ============================================================================
chain = RunnableParallel({"x": RunnablePassthrough(), "y": RunnablePassthrough()})

In [ ]:
# ============================================================================
# RUNNABLEPARALLEL: Invoke with a single string input
# ============================================================================
chain.invoke("hello")

In [ ]:
# ============================================================================
# RUNNABLEPARALLEL: Invoke with a dict input
# ============================================================================
chain.invoke({"input": "hello", "input2": "goodbye"})

In [ ]:
# ============================================================================
# RUNNABLEPARALLEL: Mix RunnablePassthrough with a plain lambda branch
# ============================================================================
chain = RunnableParallel({"x": RunnablePassthrough(), "y": lambda z: z["input2"]})

In [ ]:
# ============================================================================
# RUNNABLEPARALLEL: Invoke the mixed-branch chain
# ============================================================================
chain.invoke({"input": "hello", "input2": "goodbye"})

### Nested chains - now it gets more complicated!

In [ ]:
# ============================================================================
# NESTED CHAINS: Define a helper that extracts and uppercases a key
# ============================================================================
def find_keys_to_uppercase(input: dict):
    output = input.get("input", "not found").upper()
    return output

In [ ]:
# ============================================================================
# NESTED CHAINS: Nest a sequential chain inside a parallel branch
# ============================================================================
chain = RunnableParallel({"x": RunnablePassthrough() | RunnableLambda(find_keys_to_uppercase), "y": lambda z: z["input2"]})

In [ ]:
# ============================================================================
# NESTED CHAINS: Invoke the nested chain
# ============================================================================
chain.invoke({"input": "hello", "input2": "goodbye"})

In [ ]:
# ============================================================================
# RUNNABLEPARALLEL ASSIGN: Set up branch plus helper functions for .assign()
# ============================================================================
chain = RunnableParallel({"x": RunnablePassthrough()})

def assign_func(_):
    return 100

def multiply(input):
    return input * 10

In [ ]:
# ============================================================================
# RUNNABLEPARALLEL ASSIGN: Invoke before .assign() is applied
# ============================================================================
chain.invoke({"input": "hello", "input2": "goodbye"})

In [ ]:
# ============================================================================
# RUNNABLEPARALLEL ASSIGN: Add a computed key with .assign()
# ============================================================================
chain = RunnableParallel({"x": RunnablePassthrough()}).assign(extra=RunnableLambda(assign_func))

In [ ]:
# ============================================================================
# RUNNABLEPARALLEL ASSIGN: Invoke the chain with the assigned key
# ============================================================================
result = chain.invoke({"input": "hello", "input2": "goodbye"})
print(result)

### Combine multiple chains

In [ ]:
# ============================================================================
# COMBINE CHAINS: Build a second chain to extract and uppercase
# ============================================================================
def extractor(input: dict):
    return input.get("extra", "Key not found")

def cupper(upper: str):
    return str(upper).upper()

new_chain = RunnableLambda(extractor) | RunnableLambda(cupper)

In [ ]:
# ============================================================================
# COMBINE CHAINS: Invoke the second chain standalone
# ============================================================================
new_chain.invoke({"extra": "test"})

In [ ]:
# ============================================================================
# COMBINE CHAINS: Pipe the first chain into the second
# ============================================================================
final_chain = chain | new_chain
final_chain.invoke({"input": "hello", "input2": "goodbye"})

---
## 📝 Summary

In this notebook, we learned:

### 1. The `|` Operator Is Just Python
- `2 + 4` and `(2).__add__(4)` are the same call — Python operators are syntactic sugar for dunder methods
- A class only needs to implement `__or__` to make `a | b` work — that's exactly how LangChain's `Runnable` supports `prompt | model | parser`

### 2. A Hand-Rolled Runnable
- `CRunnable.__or__` returns a `CRunnableSequence` that stores the two composed steps and chains their `.invoke()` calls
- This mirrors (in miniature) how LangChain's real `RunnableSequence` works internally

### 3. Core LangChain Runnables
- **`RunnablePassthrough`**: returns its input unchanged — useful for preserving data alongside a transformation
- **`RunnableLambda`**: wraps any Python function so it can be composed with `|`
- **`RunnableParallel`**: runs multiple branches on the same input and collects results into a dict (a plain `{...}` dict is shorthand for it)

### 4. Nesting and Composition
- A `RunnableParallel` branch can itself be a `RunnablePassthrough() | RunnableLambda(...)` sub-chain — chains nest naturally
- `.assign()` adds a new computed key to a dict-shaped chain output without discarding the existing keys
- Two finished chains can be piped together with `|` just like any other Runnables: `final_chain = chain | new_chain`

### Next Steps
- Continue to **`3.4_LCEL_and_Runnables.ipynb`** to see these same building blocks used in a Retrieval-Augmented Generation (RAG) chain
- Revisit **`3.2_Runnables.ipynb`** for a broader tour of `RunnableBranch`, streaming, and batch processing
